# Adding Missing Prices to Original Price Dataset (Pre-Resale Merge)

This notebook updates the original property price dataset (`PriceMerge_Jun21_Sept25.csv`) by adding missing price values using the completed dataset  
(`Property_Data_Jun2021_Sep2025_MissingResaleValues_Completed.csv`).  

The pipeline ensures that missing prices in the original dataset are filled **before merging with resale data**, keeping all column names, filenames, and logic consistent with the original workflow.

The updated dataset is then ready for downstream analysis and merging with resale information.

---

### Datasets Links:
1. `PriceMerge_Jun21_Sept25.csv` (https://drive.google.com/file/d/1RUI15c-5Vi1kL1Pl8WK863Ma0ZnqEYhq/view?usp=drive_link)
2. `Property_Data_Jun2021_Sep2025_MissingResaleValues_Completed.csv` (https://drive.google.com/file/d/1p-OQFqu9F3LpDTWs3uAkrTu7otcww-qI/view?usp=drive_link)

---

### Instructions
Instruction: After cloning our repo, download the CSV files above and move it to the ds-chapa-affordable-housing/fa25-team-a/data folder, then locally run the cells below

---


### Pipeline Overview
1. Load datasets (original price dataset + completed missing prices dataset)  
2. Standardize column names to match original dataset (`Maximum Resale Price`)  
3. Merge datasets to align missing prices  
4. Identify and optionally review addresses where old and new prices differ  
5. Fill missing prices only where NA in the original dataset  
6. Save the updated price dataset ready for resale merge


In [ ]:
# ============================================================
# CHAPA Data Integration Pipeline
# Step: Combine Original Price Data with Completed Missing Prices
# ============================================================

"""
This script updates the original CHAPA price dataset with the completed
missing prices data before merging it with the resale dataset.

Reference:
    - Parsed Resale Data: ParsedResale_updatedSept2025_New.csv
    - Parsed Price Data: PriceMerge_Jun21_Sept25.csv
"""

import pandas as pd

# ------------------------------------------------------------
# 1. Load Datasets
# ------------------------------------------------------------
def load_datasets():
    """Load original price and new completed price datasets."""
    original_price = pd.read_csv("../data/PriceMerge_Jun21_Sept25.csv")
    new_price = pd.read_csv("../data/Property_Data_Jun2021_Sep2025_MissingResaleValues_Completed.csv")
    return original_price, new_price


# ------------------------------------------------------------
# 2. Standardize Column Names
# ------------------------------------------------------------
def standardize_columns(new_price):
    """Ensure the price column matches the original dataset naming convention."""
    new_price = new_price.rename(columns={"Price": "Maximum Resale Price"})
    return new_price


# ------------------------------------------------------------
# 3. Merge Price Data
# ------------------------------------------------------------
def merge_price_data(original_price, new_price):
    """
    Merge original price dataset with new completed price data.
    Fills missing 'Maximum Resale Price' values where available in new data.
    """
    keys = ["Town", "Address", "Unit Number"]

    merged = original_price.merge(
        new_price[keys + ["Maximum Resale Price"]],
        on=keys,
        how="left",
        suffixes=("", "_new")
    )

    # Identify cases where both old and new price exist but differ
    overwritten = merged[
        merged["Maximum Resale Price"].notna() &
        merged["Maximum Resale Price_new"].notna() &
        (merged["Maximum Resale Price"] != merged["Maximum Resale Price_new"])
    ]

    if not overwritten.empty:
        print("⚠️ Properties with overwritten resale prices:")
        print(overwritten[["Town", "Address", "Unit Number"]])
        overwritten[["Town", "Address", "Unit Number"]].to_csv("overwritten_addresses.csv", index=False)
    else:
        print("✅ No overwrites detected.")

    # Fill missing price values only where NA
    merged["Maximum Resale Price"] = merged["Maximum Resale Price"].fillna(merged["Maximum Resale Price_new"])

    # Drop helper column
    merged.drop(columns=["Maximum Resale Price_new"], inplace=True)
    return merged


# ------------------------------------------------------------
# 4. Save Updated Dataset
# ------------------------------------------------------------
def save_dataset(df, filename="PriceData_Filled.csv"):
    """Save the updated price dataset."""
    df.to_csv(filename, index=False)
    print(f"✅ Missing prices filled and saved as {filename}")


# ------------------------------------------------------------
# 5. Pipeline Execution
# ------------------------------------------------------------
def main():
    print("🚀 Starting price data integration pipeline...")
    original_price, new_price = load_datasets()
    new_price = standardize_columns(new_price)
    updated_price = merge_price_data(original_price, new_price)
    save_dataset(updated_price)
    print("🏁 Price data update complete. Ready for resale merge step.")


if __name__ == "__main__":
    main()
